# 財經時序實作｜用 Transformer 預測下一日報酬率

每個交易日收盤後，用最近 20 個已知報酬率預測下一個交易日的報酬率。同一批測試日期上，比較小型 Transformer 與每天預測 0% 的基準。

先用手算核對滑動視窗，再在 CPU 跑完整流程。預設固定種子的合成資料可直接重跑。要分析加權股價指數，可匯入從證交所下載的月報 CSV。雲端初次安裝套件需要網路。

[閱讀講義](https://github.com/Samuel-ncu/AI-Stastic/blob/main/docs/financial-time-series-transformer.md) · [在 Colab 執行](https://colab.research.google.com/github/Samuel-ncu/AI-Stastic/blob/main/notebooks/financial_time_series_transformer.ipynb) · [證交所加權指數歷史資料](https://www.twse.com.tw/zh/indices/taiex/mi-5min-hist.html)

In [ ]:
%pip -q install numpy matplotlib "torch>=2.5,<3"

## 1. 從六個收盤指數做出兩筆樣本

用 $r_t=100(P_t/P_{t-1}-1)$ 把報酬率以百分比數值記錄。例如 $r_2=2.00$ 代表上漲 2.00%。第 1 天缺前一日收盤值，從第 2 天開始計算。訓練使用未四捨五入的數值。

| 交易日 | 收盤指數 $P_t$ | 報酬率 |
|---:|---:|---:|
| 1 | 100 | — |
| 2 | 102 | +2.00% |
| 3 | 101 | −0.98% |
| 4 | 103 | +1.98% |
| 5 | 104 | +0.97% |
| 6 | 102 | −1.92% |

輸入前 3 筆報酬率就得到兩筆樣本。第 1 筆用第 2～4 日預測第 5 日，第 2 筆用第 3～5 日預測第 6 日。完整實作把輸入延長到 20 個交易日。

In [ ]:
import csv
from datetime import date
from io import StringIO
from pathlib import Path
from copy import deepcopy

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

def percent_returns(closes):
    closes = np.asarray(closes, dtype=np.float64)
    if not np.isfinite(closes).all() or np.any(closes <= 0):
        raise ValueError("Close values must be finite and positive.")
    return 100 * (closes[1:] / closes[:-1] - 1)

def next_day_windows(values, lookback):
    windows = np.lib.stride_tricks.sliding_window_view(values, lookback + 1)
    return windows[:, :-1], windows[:, -1]

example_returns = percent_returns([100, 102, 101, 103, 104, 102])
example_x, example_y = next_day_windows(example_returns, lookback=3)
print("Daily returns (%):", example_returns.round(2))
print("X (%):\n", example_x.round(2))
print("y (%):", example_y.round(2))
np.testing.assert_allclose(example_x.round(2), [[2.00, -0.98, 1.98], [-0.98, 1.98, 0.97]])
np.testing.assert_allclose(example_y.round(2), [0.97, -1.92])

## 2. 選擇資料來源

- **直接執行**。保留 `data_source = "synthetic"`。合成資料有固定的隨機種子及已知的前一期依賴，供練習核對流程。
- **使用實際加權指數**。從[證交所歷史資料](https://www.twse.com.tw/zh/indices/taiex/mi-5min-hist.html)逐月下載一段連續期間的 CSV，例如連續兩年。Colab 可從左側「檔案」上傳。把下方 `data_source` 改為 `"twse"`，並依上傳位置設定 `csv_pattern`，例如 `"/content/MI_5MINS_HIST*.csv"`。如果檔名不同，改成實際檔名或相符的樣式。保留原始 `日期` 與 `收盤指數` 欄名。

程式會轉換民國年月日、移除指數的千分位符號，並把各月依日期合併。下載時保持月份連續，程式依 CSV 中相鄰的兩個觀測日計算報酬率。

In [ ]:
from glob import glob

data_source = "synthetic"
csv_pattern = "/content/MI_5MINS_HIST*.csv"
lookback = 20
seed = 42

def parse_twse_csv(content):
    date_col = close_col = None
    closes_by_date = {}
    for values in csv.reader(StringIO(content)):
        fields = [value.strip().lstrip("\ufeff") for value in values]
        if "日期" in fields and "收盤指數" in fields:
            date_col = fields.index("日期")
            close_col = fields.index("收盤指數")
            continue
        if date_col is None or len(fields) <= max(date_col, close_col):
            continue
        parts = fields[date_col].split("/")
        if len(parts) != 3 or not all(part.isdecimal() for part in parts):
            continue
        year, month, day = map(int, parts)
        trading_day = date(year + 1911 if year < 1900 else year, month, day)
        close = float(fields[close_col].replace(",", ""))
        if trading_day in closes_by_date and closes_by_date[trading_day] != close:
            raise ValueError(f"Conflicting closing values for {trading_day}.")
        closes_by_date[trading_day] = close
    if not closes_by_date:
        raise ValueError("Expected CSV rows with 日期 and 收盤指數.")
    return closes_by_date

def read_twse_closes(pattern):
    files = sorted(glob(pattern))
    if not files:
        raise FileNotFoundError(f"No TWSE CSV file matches: {pattern}")
    closes_by_date = {}
    for name in files:
        file_bytes = Path(name).read_bytes()
        try:
            content = file_bytes.decode("utf-8-sig")
        except UnicodeDecodeError:
            content = file_bytes.decode("cp950")
        for trading_day, close in parse_twse_csv(content).items():
            if trading_day in closes_by_date and closes_by_date[trading_day] != close:
                raise ValueError(f"Conflicting closing values for {trading_day}.")
            closes_by_date[trading_day] = close
    trading_days = sorted(closes_by_date)
    prices = [closes_by_date[day] for day in trading_days]
    return np.array(trading_days, dtype="datetime64[D]"), np.array(prices)

# 檢查教材會用到的民國日期、千分位與抬頭格式。
csv_example = '示例\n日期,開盤指數,最高指數,最低指數,收盤指數\n113/01/02,100,104,98,"17,100.25"'
assert parse_twse_csv(csv_example) == {date(2024, 1, 2): 17100.25}

if data_source == "synthetic":
    rng = np.random.default_rng(seed)
    simulated_returns = np.empty(459)
    previous_return = 0.0
    for i, shock in enumerate(rng.normal(0, 0.008, size=len(simulated_returns))):
        previous_return = 0.45 * previous_return + shock
        simulated_returns[i] = previous_return
    closes = 100 * np.r_[1, np.cumprod(1 + simulated_returns)]
    dates = np.busday_offset("2023-01-02", np.arange(len(closes)))
elif data_source == "twse":
    dates, closes = read_twse_closes(csv_pattern)
else:
    raise ValueError(f"Choose synthetic or twse, got {data_source!r}.")

returns = percent_returns(closes)
return_dates = dates[1:]
print(f"{data_source}: {len(closes)} observations, {dates[0]} to {dates[-1]}")

## 3. 用答案日期做時間切分

前 60% 報酬日為訓練期，中間 20% 為驗證期，最後 20% 為測試期。每一筆依**要預測的那一天**分到集合。第一個測試視窗可以使用測試日之前已經知道的報酬率。這是在模擬每天收盤後重做一次次日預測。

標準化參數只由訓練期報酬率估計。用同一組 $m,s_0$ 處理各期，把模型輸出的 $z$ 還原成報酬率時也用同一組值。若換成多步預測，需要重新界定起報日當時已知的輸入。

In [ ]:
train_end = int(0.60 * len(returns))
valid_end = int(0.80 * len(returns))
if not 0 < lookback < train_end < valid_end < len(returns):
    raise ValueError("Use a shorter lookback or load more observations.")
input_mean = returns[:train_end].mean()
input_scale = returns[:train_end].std()
if not np.isfinite(input_scale) or input_scale == 0:
    raise ValueError("The training returns need a finite, nonzero standard deviation.")

scaled = (returns - input_mean) / input_scale
inputs, labels = next_day_windows(scaled, lookback)
target_positions = np.arange(lookback, len(returns))
target_dates = return_dates[target_positions]
last_input_dates = return_dates[target_positions - 1]
train_mask = target_positions < train_end
valid_mask = (target_positions >= train_end) & (target_positions < valid_end)
test_mask = target_positions >= valid_end

assert np.all(last_input_dates < target_dates)
assert target_dates[train_mask][-1] < target_dates[valid_mask][0]
assert target_dates[valid_mask][-1] < target_dates[test_mask][0]
np.testing.assert_allclose(labels[test_mask] * input_scale + input_mean, returns[valid_end:], atol=1e-12)

x = torch.tensor(inputs[..., None], dtype=torch.float32)
y = torch.tensor(labels, dtype=torch.float32)
for name, mask in [("Train", train_mask), ("Validation", valid_mask), ("Test", test_mask)]:
    print(f"{name:>10}: {mask.sum():3d} targets, {target_dates[mask][0]} to {target_dates[mask][-1]}")

first_test = np.flatnonzero(test_mask)[0]
print("First test input ends:", last_input_dates[first_test])
print("First test target date:", target_dates[first_test])
print("X (first test window, %):", (inputs[first_test] * input_scale + input_mean).round(3))
print(f"y (first test window, %): {returns[target_positions[first_test]]:.3f}")

## 4. 每天是一個數值向量，模型輸出一個數字

`Linear` 把每個標準化報酬率投影成 16 維向量，再加上位置向量，讓模型能區分哪些是較近或較早的資料。一層雙頭自注意力讓視窗內的 20 個已觀測日互相參考。取最後一日的表示，映射成下一日的預測值。

一個視窗只對最後再往前預測一天，輸入日期都已觀測，所以視窗內可彼此參考。如果改成在窗內每個位置都預測自己的下一日，就要用因果遮罩，保護每個位置的時間邊界。

用標準化尺度上的平均平方誤差（MSE）訓練。每個 epoch 結束後紀錄訓練與驗證損失，再選驗證 MSE 最低的權重。訓練視窗可以打亂分批，視窗裡的時間順序仍保留。

In [ ]:
class ReturnTransformer(nn.Module):
    def __init__(self, lookback):
        super().__init__()
        self.input_projection = nn.Linear(1, 16)
        self.positions = nn.Parameter(torch.zeros(1, lookback, 16))
        self.encoder = nn.TransformerEncoderLayer(
            d_model=16, nhead=2, dim_feedforward=32, dropout=0.1, batch_first=True
        )
        self.readout = nn.Linear(16, 1)

    def forward(self, windows):
        features = self.input_projection(windows) + self.positions
        encoded = self.encoder(features)
        return self.readout(encoded[:, -1, :]).squeeze(-1)

torch.set_num_threads(1)
torch.manual_seed(seed)
model = ReturnTransformer(lookback)
loss_function = nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001)
train_batches = DataLoader(
    TensorDataset(x[train_mask], y[train_mask]),
    batch_size=32,
    shuffle=True,
    generator=torch.Generator().manual_seed(seed),
)
epochs = 20
train_losses, valid_losses = [], []
best_valid_loss = float("inf")
best_state = None

for epoch in range(1, epochs + 1):
    model.train()
    for batch_x, batch_y in train_batches:
        optimizer.zero_grad()
        loss = loss_function(model(batch_x), batch_y)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.inference_mode():
        train_losses.append(loss_function(model(x[train_mask]), y[train_mask]).item())
        valid_losses.append(loss_function(model(x[valid_mask]), y[valid_mask]).item())
    if valid_losses[-1] < best_valid_loss:
        best_valid_loss = valid_losses[-1]
        best_epoch = epoch
        best_state = deepcopy(model.state_dict())

model.load_state_dict(best_state)
print(f"Best validation epoch: {best_epoch}, MSE: {best_valid_loss:.4f}")

## 5. 把預測換回報酬率，和 0% 基準比較

現在才讀測試結果。將模型輸出還原為報酬率，計算 MAE 和 RMSE，單位是百分點。例如真值為 +0.97%、預測是 +0.40%，絕對誤差是 0.57 個百分點。若一律預測 0%，同一筆的絕對誤差是 0.97 個百分點。

兩個方法使用完全相同的測試日期，才有可比性。訓練或驗證損失降低，只回答模型在那些資料上的學習情況。測試分數用來判讀沒有用在擬合或選設定的期間。

In [ ]:
model.eval()
with torch.inference_mode():
    predicted = model(x[test_mask]).numpy() * input_scale + input_mean

test_dates = target_dates[test_mask]
actual = returns[target_positions[test_mask]]
zero_baseline = np.zeros_like(actual)
assert predicted.shape == actual.shape
assert np.isfinite(predicted).all()

print(f"{'Method':<14} {'MAE (pp)':>10} {'RMSE (pp)':>11}")
for name, forecast in [("Zero return", zero_baseline), ("Transformer", predicted)]:
    errors = actual - forecast
    mae = np.abs(errors).mean()
    rmse = np.sqrt(np.square(errors).mean())
    print(f"{name:<14} {mae:>10.4f} {rmse:>11.4f}")

fig, axes = plt.subplots(2, 1, figsize=(10, 7), layout="constrained")
axes[0].plot(range(1, epochs + 1), train_losses, label="Train")
axes[0].plot(range(1, epochs + 1), valid_losses, label="Validation")
axes[0].axvline(best_epoch, color="0.45", linestyle=":", label="Selected epoch")
axes[0].set(xlabel="Epoch", ylabel="MSE (standardized units)", title="Learning curves")
axes[0].legend()

axes[1].plot(test_dates, actual, color="0.38", linewidth=1, label="Observed")
axes[1].plot(test_dates, predicted, color="#1565a8", linewidth=1.2, label="Transformer")
axes[1].axhline(0, color="#a64b23", linestyle="--", label="Zero-return baseline")
axes[1].set(xlabel="Target date", ylabel="Return (%)", title="One-step forecasts on the held-out dates")
axes[1].legend()
plt.show()

## 作業與檢核

1. 抄下第一個測試視窗最後的輸入日期、答案日期與兩個模型的 MAE，說明這報酬率在做預測時是否已知。
2. 用同一組訓練、驗證時間區間，比較 5 日和 20 日視窗。用驗證損失選一種設定，再對保留的測試期間報告一次結果。記錄結果若落後 0% 基準，代表這段期間的平均誤差有何差異。
3. 如改用實際資料，附上來源、涵蓋日期與見到的缺漏。解釋合成系列為何可能較容易學到固定的前一期關係。

若題目改成預測下一日是否上漲，可用 $y=1$ 表示報酬率大於 0，其餘標 0。把輸出解讀成分類分數並搭配 BCE，回看[第 05 課](https://github.com/Samuel-ncu/AI-Stastic/blob/main/docs/units/05-statistical-models-loss.md)。